# Tren Crypto
### Chien luoc lien quan ML, MACD, RSI:
### Toi uu hoa Feature de tim ra feature tot nhat tu dau nam den gio, de dua vao chien luoc o tren
### Xay dung OG de chay tu dong

# Load data

In [2]:
import sys
sys.path.append("../Common")
import CommonBinance

# Toi uu Feature => Target

In [5]:
symbol = "BTCUSDT"
from_date = "2025-01-01"
to_date = "2025-04-03"
timeframe = "1d"
data = CommonBinance.CommonBinance.loaddataBinance_FromTo(symbol, from_date, to_date, timeframe)
data

,Datetime,Open,High,Low,Close,Volume
0,2025-01-01,93576.00,95151.15,92888.00,94591.79,10373.32613
1,2025-01-02,94591.78,97839.50,94392.00,96984.79,21970.48948
2,2025-01-03,96984.79,98976.91,96100.01,98174.18,15253.82936
3,2025-01-04,98174.17,98778.43,97514.79,98220.50,8990.05651
4,2025-01-05,98220.51,98836.85,97276.79,98363.61,8095.63723
...,...,...,...,...,...,...
88,2025-03-30,82648.53,83534.64,81565.00,82389.99,9864.49508
89,2025-03-31,82390.00,83943.08,81278.52,82550.01,20569.13885
90,2025-04-01,82550.00,85579.46,82432.74,85158.34,20190.39697
91,2025-04-02,85158.35,88500.00,82320.00,82516.29,39931.45700


In [9]:
import talib
data['MACD'] = talib.MACD(data['Close'], fastperiod=12, slowperiod=26, signalperiod=9)[0]
data['MACD_Signal'] = talib.MACD(data['Close'], fastperiod=12, slowperiod=26, signalperiod=9)[1]
data['MACD_Hist'] = talib.MACD(data['Close'], fastperiod=12, slowperiod=26, signalperiod=9)[2]
data['RSI'] = talib.RSI(data['Close'], timeperiod=14)

data

,Datetime,Open,High,Low,Close,Volume,MACD,MACD_Signal,MACD_Hist,RSI
0,2025-01-01,93576.00,95151.15,92888.00,94591.79,10373.32613,NaN,NaN,NaN,NaN
1,2025-01-02,94591.78,97839.50,94392.00,96984.79,21970.48948,NaN,NaN,NaN,NaN
2,2025-01-03,96984.79,98976.91,96100.01,98174.18,15253.82936,NaN,NaN,NaN,NaN
3,2025-01-04,98174.17,98778.43,97514.79,98220.50,8990.05651,NaN,NaN,NaN,NaN
4,2025-01-05,98220.51,98836.85,97276.79,98363.61,8095.63723,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
88,2025-03-30,82648.53,83534.64,81565.00,82389.99,9864.49508,-1005.454659,-1144.634737,139.180078,41.870740
89,2025-03-31,82390.00,83943.08,81278.52,82550.01,20569.13885,-1106.424716,-1136.992733,30.568017,42.303385
90,2025-04-01,82550.00,85579.46,82432.74,85158.34,20190.39697,-964.851572,-1102.564501,137.712929,48.970418
91,2025-04-02,85158.35,88500.00,82320.00,82516.29,39931.45700,-1053.698807,-1092.791362,39.092555,43.488642


In [10]:
# Import các thư viện cần thiết
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [11]:
import numpy as np
from itertools import combinations
import talib

# Loai bo các hàng có giá trị NaN trong dữ liệu
# data.dropna(inplace=True)

# Thay thế NaN bằng giá trị trung bình của cột
data.fillna(data.mean(), inplace=True)

# Khởi tạo biến mục tiêu 'y' là giá đóng cửa
y = data['Close']

# Định nghĩa cơ bản của các đặc trưng
# 'Open', 'High', 'Low', 'Volume', 'Momentum', 'RSI'
basic_features = ['Open', 'High', 'Low', 'Volume', 'MACD', 'RSI']

# Tạo tất cả tổ hợp có thể từ basic_features
# Sử dụng itertools để tạo tất cả các tổ hợp của các đặc trưng
features_combinations = []
for r in range(1, len(basic_features) + 1):
    for combo in combinations(basic_features, r):
        features_combinations.append(list(combo))

# Luu trữ kết quả MSE cho từng tổ hợp
mse_scores = {}

# Kiểm tra từng kết hợp đặc trưng
for features in features_combinations:
    X = data[features].copy() # Copy dữ liệu để tránh thay đổi gốc
	# Thay thế NaN bằng giá trị trung bình của cột
    X.fillna(X.mean(), inplace=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = DecisionTreeRegressor(random_state=42)
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    
    mse_scores[str(features)] = mse

# Tìm kết hợp đặc trưng với MSE thấp nhất
best_features = min(mse_scores, key=mse_scores.get)
print(f"Best features: {best_features} with MSE: {mse_scores[best_features]}")

Best features: ['Open', 'High', 'Low', 'Volume', 'MACD'] with MSE: 1794618.3587473661


# Tu dong hoa OG